# 📖 Data story: the *Arabidopsis* spaceflight response across studies

In the [dataset explorer](OSDR_dataset_explorer.ipynb) we drilled into a single
study (OSD-120) and watched one gene respond to spaceflight. That raises the
question every scientist asks next:

> **Is the response *reproducible*? Do independent *Arabidopsis* spaceflight
> experiments agree — and does spaceflight reach beyond the transcriptome into
> the *epigenome*?**

This chapter strings the explorer's per-dataset outputs into a cross-study
narrative. We:

1. assemble a **panel of independent *Arabidopsis* spaceflight RNA-seq studies**,
2. compute one common metric — the **spaceflight log₂ fold-change** per gene,
3. ask how well the studies **agree**, and which genes form a **reproducible core**,
4. then meet the **DNA-methylation (epigenetic) studies** that add the next layer.

> 🚀 Open this page in Binder/Colab (rocket icon) to re-run and extend the story.

In [1]:
import io, re
import numpy as np
import pandas as pd
import requests
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

API = "https://visualization.osdr.nasa.gov/biodata/api/v2"
GEODE = "https://osdr.nasa.gov/geode-py/ws/studies"

# Independent Arabidopsis spaceflight RNA-seq studies with processed counts
PANEL = ["OSD-120", "OSD-37", "OSD-38", "OSD-321", "OSD-281", "OSD-427", "OSD-217"]

## 1. The cast — independent spaceflight experiments

Each of these is a separate *Arabidopsis* spaceflight transcriptomics study from a
different team, mission, and year — the perfect test of reproducibility.

In [2]:
def normalized_counts(acc):
    """Download a study's GeneLab normalized-counts table."""
    d = requests.get(f"{API}/dataset/{acc}/files/", timeout=60).json()
    names = list(d[list(d)[0]].get("files", d[list(d)[0]]))
    f = next(n for n in names if "Normalized_Counts" in n and n.endswith(".csv") and "rRNArm" not in n)
    return pd.read_csv(f"{GEODE}/{acc}/download?source=datamanager&file={f}", index_col=0)

def flight_ground(acc, cols):
    """Map each sample column to 'Spaceflight' / 'Ground' using the study's factor values."""
    url = f"{API}/query/metadata/?id.accession={acc}&id.sample name&study.factor value&format=csv"
    df = pd.read_csv(io.StringIO(requests.get(url, timeout=60).text)).drop_duplicates("id.sample name")
    fcols = [c for c in df.columns if c.startswith("study.factor value")]
    sf = next((c for c in fcols if re.search(r"(?i)spaceflight|space.?flight", c)), None)
    labels = {}
    if sf:
        m = df.set_index("id.sample name")[sf].astype(str)
        for c in cols:
            v = m.get(c, "").lower()
            labels[c] = "Spaceflight" if ("flight" in v or "space" in v) else ("Ground" if "ground" in v else None)
    # fallback: parse the GeneLab sample name itself
    for c in cols:
        if labels.get(c) is None:
            labels[c] = "Spaceflight" if "_FLT_" in c else ("Ground" if "_GC_" in c else None)
    return labels

cast = []
for acc in PANEL:
    meta = requests.get(f"{API}/dataset/{acc}/", timeout=60).json()[acc]["metadata"]
    title = meta.get("study title", "")
    title = title[0] if isinstance(title, list) else title
    cast.append({"accession": acc, "title": title[:70]})
cast = pd.DataFrame(cast)
cast

,accession,title
0,OSD-120,Genetic dissection of the Arabidopsis spacefli...
1,OSD-37,Comparison of the spaceflight transcriptome of...
2,OSD-38,Proteomics and Transcriptomics analysis of Ara...
3,OSD-321,Relevance of Unfolded Protein Response to Spac...
4,OSD-281,"During development, the Sku5 mutant roots enga..."
5,OSD-427,Characterizing Epigenetic Changes in Methylati...
6,OSD-217,Characterization of Epigenetic Regulation in a...


## 2. One common metric: the spaceflight log₂ fold-change

For every study we compute, per gene,
$\log_2\!\frac{\text{mean expression in spaceflight}}{\text{mean expression on the ground}}$.
Positive = up in space, negative = down. Putting every study on this same scale
lets us compare them directly.

In [3]:
def spaceflight_lfc(acc):
    counts = normalized_counts(acc)
    lab = flight_ground(acc, counts.columns)
    flt = [c for c in counts.columns if lab.get(c) == "Spaceflight"]
    grd = [c for c in counts.columns if lab.get(c) == "Ground"]
    if len(flt) < 2 or len(grd) < 2:
        return None, len(flt), len(grd)
    lfc = np.log2((counts[flt].mean(axis=1) + 1) / (counts[grd].mean(axis=1) + 1))
    return lfc.rename(acc), len(flt), len(grd)

series, summary = {}, []
for acc in PANEL:
    lfc, nf, ng = spaceflight_lfc(acc)
    summary.append({"accession": acc, "flight": nf, "ground": ng,
                    "used": lfc is not None})
    if lfc is not None:
        series[acc] = lfc
lfc_mat = pd.DataFrame(series).dropna()          # genes (rows) x studies (cols)
print(f"{lfc_mat.shape[1]} studies compared over {lfc_mat.shape[0]} shared genes")
pd.DataFrame(summary)

7 studies compared over 21527 shared genes


,accession,flight,ground,used
0,OSD-120,18,18,True
1,OSD-37,28,28,True
2,OSD-38,3,9,True
3,OSD-321,22,22,True
4,OSD-281,16,16,True
5,OSD-427,24,24,True
6,OSD-217,6,6,True


## 3. Do independent studies agree?

If the spaceflight response is real and reproducible, studies should **correlate**:
genes that go up in one experiment should tend to go up in the others.

In [4]:
corr = lfc_mat.corr()
fig = px.imshow(corr, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1,
                title="Cross-study agreement of the spaceflight response (log₂FC correlation)")
fig.update_layout(height=480)
fig.show()

In [5]:
# Look at the single best-correlated pair head-to-head
import itertools
pairs = list(itertools.combinations(lfc_mat.columns, 2))
a, b = max(pairs, key=lambda p: lfc_mat[p[0]].corr(lfc_mat[p[1]]))
r = lfc_mat[a].corr(lfc_mat[b])
fig = px.scatter(lfc_mat, x=a, y=b, opacity=0.25,
                 title=f"Gene-by-gene spaceflight response: {a} vs {b}  (Pearson r = {r:.2f})",
                 labels={a: f"log₂FC ({a})", b: f"log₂FC ({b})"})
fig.update_layout(height=480)
fig.show()

## 4. The reproducible core

Which genes respond to spaceflight **consistently** — same direction, strong
effect — across the panel? Those are the robust core of the *Arabidopsis*
spaceflight transcriptome.

In [6]:
mean_lfc = lfc_mat.mean(axis=1)
sign_agreement = np.sign(lfc_mat).eq(np.sign(mean_lfc), axis=0).mean(axis=1)
core = mean_lfc[sign_agreement >= 0.8].abs().sort_values(ascending=False).head(25).index

fig = px.imshow(lfc_mat.loc[core], aspect="auto", color_continuous_scale="RdBu_r",
                zmin=-3, zmax=3,
                labels={"x": "study", "y": "gene (AGI locus)", "color": "log₂FC"},
                title="Reproducible spaceflight-responsive genes (consistent across studies)")
fig.update_layout(height=620)
fig.show()

## 5. The next layer: spaceflight and the *epigenome*

Spaceflight doesn't only change which genes are *expressed* — it can change how
the genome is **chemically marked**. OSDR holds *Arabidopsis*
**DNA-methylation (bisulfite-sequencing)** studies that probe this epigenetic
layer:

In [7]:
EPI = ["OSD-217", "OSD-220", "OSD-416"]
rows = []
for acc in EPI:
    meta = requests.get(f"{API}/dataset/{acc}/", timeout=60).json()[acc]["metadata"]
    def show(k):
        v = meta.get(k, "")
        return ", ".join(map(str, v)) if isinstance(v, list) else v
    rows.append({"accession": acc, "title": show("study title")[:55],
                 "assays": show("study assay technology type"),
                 "factors": show("study factor type")})
pd.DataFrame(rows)

,accession,title,assays,factors
0,OSD-217,Characterization of Epigenetic Regulation in a...,"Whole Genome Bisulfite Sequencing, RNA Sequenc...","Space Flight, organism part"
1,OSD-220,Methylome Analysis of Arabidopsis Seedlings Ex...,Whole Genome Bisulfite Sequencing,Space Flight
2,OSD-416,Characterizing Epigenetic Changes in Methylati...,Bisulfite sequencing,"Space Flight, Genotype, Organism Part"


```{admonition} A multi-omic opportunity
:class: tip
**OSD-217 measures *both* DNA methylation *and* RNA-seq on the same spaceflight
experiment** — and its transcriptome is already in our panel above. That pairing
lets researchers ask the deeper question: *are the spaceflight-responsive genes
also differentially methylated?*

The biodata API exposes these studies' raw bisulfite data (processed
methylation tables aren't served through the API yet), so a quantitative
methylation-vs-expression analysis is the natural **frontier** this story points
to — the data is there in OSDR, waiting.
```

## 6. The story so far

- The *Arabidopsis* spaceflight transcriptome is **not a one-study fluke** — independent
  experiments **positively correlate**, and a **reproducible core** of genes responds
  in a consistent direction.
- Differences between studies (ecotype, organ, light, mission) add scatter — which is
  itself a story about *context-dependence*.
- Spaceflight reaches **beyond expression into the epigenome**; OSDR's methylation
  studies (especially the multi-omic **OSD-217**) are the next layer to mine.

Every figure here was built live from the OSDR API — change `PANEL`, swap in other
organisms, or join the methylation data to write the next chapter.